In [2]:
# ============================================================
# STAGE 1: Train Restaurant-only checkpoint
# ============================================================
!pip install transformers datasets evaluate seqeval -q

import os, json, gc
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
import evaluate
from transformers import (
    AutoModelForTokenClassification, AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorForTokenClassification
)

seqeval = evaluate.load("seqeval")

from google.colab import drive
drive.mount('/content/drive')
df = pd.read_parquet("/content/drive/MyDrive/MABSA_Shared_Project/data/master_dataset.parquet")
print("df loaded:", df.shape)

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

# --- Helpers ---
def make_bio_tags(sentence, aspect, category, sentiment, task="tasd"):
    tokens = sentence.split()
    tags = ["O"] * len(tokens)
    if pd.isna(aspect) or str(aspect).strip() in ("", "NULL"):
        return tokens, tags
    aspect_tokens = aspect.split()
    n = len(aspect_tokens)
    for i in range(len(tokens) - n + 1):
        if tokens[i:i+n] == aspect_tokens:
            tags[i] = f"B-{category}-{sentiment}"
            for j in range(1, n):
                tags[i+j] = f"I-{category}-{sentiment}"
            break
    return tokens, tags

def build_tagged_df(source_df, task="tasd"):
    records = []
    for _, row in source_df.iterrows():
        tokens, tags = make_bio_tags(row.sentence, row.aspect, row.category, row.sentiment, task=task)
        records.append({"tokens": tokens, "tags": tags, "split": row.split})
    return pd.DataFrame(records)

def make_tokenize_fn(label2id_local):
    def tokenize_and_align_labels(examples):
        tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
        all_labels = []
        for i, tags in enumerate(examples["tags"]):
            word_ids = tokenized_inputs.word_ids(batch_index=i)
            label_ids, prev = [], None
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != prev:
                    label_ids.append(label2id_local[tags[word_idx]])
                else:
                    label_ids.append(-100)
                prev = word_idx
            all_labels.append(label_ids)
        tokenized_inputs["labels"] = all_labels
        return tokenized_inputs
    return tokenize_and_align_labels

def make_compute_metrics(label_list_local):
    def compute_metrics(p):
        preds = np.argmax(p.predictions, axis=2)
        true_labels = [[label_list_local[l] for l in label if l != -100] for label in p.label_ids]
        true_preds = [[label_list_local[pp] for (pp, l) in zip(pred, label) if l != -100]
                      for pred, label in zip(preds, p.label_ids)]
        r = seqeval.compute(predictions=true_preds, references=true_labels)
        return {"f1": r["overall_f1"], "precision": r["overall_precision"], "recall": r["overall_recall"]}
    return compute_metrics

# --- Build Restaurant-only data ---
restaurant_en = df[(df.domain == "restaurant") & (df.language == "en")].copy()
print("Restaurant rows:", restaurant_en.shape)

sentiments = ["positive", "negative", "neutral"]
restaurant_categories = sorted(restaurant_en["category"].dropna().unique())
restaurant_label2id = {l: i for i, l in enumerate(
    ["O"] + [f"{p}-{c}-{s}" for c in restaurant_categories for s in sentiments for p in ["B", "I"]]
)}
print("Restaurant-only labels:", len(restaurant_label2id))

restaurant_tagged = build_tagged_df(restaurant_en, task="tasd")
restaurant_tok_fn = make_tokenize_fn(restaurant_label2id)

train_ds = Dataset.from_pandas(restaurant_tagged[restaurant_tagged.split == "train"][["tokens", "tags"]]).map(restaurant_tok_fn, batched=True)
val_ds = Dataset.from_pandas(restaurant_tagged[restaurant_tagged.split == "dev"][["tokens", "tags"]]).map(restaurant_tok_fn, batched=True)

# --- Train ---
collator = DataCollatorForTokenClassification(tokenizer)
model = AutoModelForTokenClassification.from_pretrained("xlm-roberta-base", num_labels=len(restaurant_label2id))

args = TrainingArguments(
    output_dir="/content/xlmr_restaurant_only",
    eval_strategy="epoch",
    save_strategy="no",              # standing rule
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    weight_decay=0.01,
    logging_steps=50,
    fp16=True,
)

trainer_restaurant = Trainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer, data_collator=collator,
    compute_metrics=make_compute_metrics(list(restaurant_label2id.keys())),
)
trainer_restaurant.train()

# --- Save + verify ---
out_dir = "/content/drive/MyDrive/MABSA_Shared_Project/models/xlmr_restaurant_only_final"
os.makedirs(out_dir, exist_ok=True)
trainer_restaurant.save_model(out_dir)
tokenizer.save_pretrained(out_dir)
with open(os.path.join(out_dir, "label_map.json"), "w") as f:
    json.dump(restaurant_label2id, f)

check = AutoModelForTokenClassification.from_pretrained(out_dir)
print("Saved + verified:", check.config.num_labels == len(restaurant_label2id))

# --- Clear GPU before next run ---
del model, trainer_restaurant, check
gc.collect()
torch.cuda.empty_cache()
print("GPU cleared. Restart runtime before Stage 2 if memory doesn't drop.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.1 MB/s eta 0:00:00


Mounted at /content/drive
df loaded: (438341, 8)


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.384410,0.365962,0.133768,0.194313,0.101990
2,0.288705,0.319891,0.238908,0.380435,0.174129
3,0.254125,0.297610,0.272048,0.340824,0.226368
4,0.218890,0.283733,0.309091,0.323370,0.296020
5,0.201737,0.268356,0.335150,0.370482,0.305970
6,0.172121,0.262126,0.373599,0.374065,0.373134
7,0.157625,0.249717,0.386667,0.416667,0.360697
8,0.154579,0.255837,0.381071,0.381546,0.380597
9,0.153824,0.248951,0.408867,0.404878,0.412935
10,0.136043,0.252337,0.429612,0.419431,0.440299


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Saved + verified: True
GPU cleared. Restart runtime before Stage 2 if memory doesn't drop.


In [7]:
print("Restaurant-only sentiments:", sorted(df[(df.domain=="restaurant")&(df.language=="en")]["sentiment"].dropna().unique()))
print("All domains sentiments:", sorted(df["sentiment"].dropna().unique()))

Restaurant-only sentiments: ['negative', 'neutral', 'positive']
All domains sentiments: ['NEG', 'NEU', 'NEU ', 'Neg', 'Neu', 'POS', 'conflict', 'negative', 'neutral', 'positive']


In [9]:
# Check which domains actually contain the messy labels first
print(rhf_en.groupby("domain")["sentiment"].unique())

domain
food          [negative, neutral, positive, conflict]
hotel         [positive, negative, neutral, conflict]
restaurant              [negative, positive, neutral]
Name: sentiment, dtype: object


In [11]:
# ============================================================
# Stage 2: Build Restaurant+Hotel+Food tagged dataset
# ============================================================
rhf_en = df[(df.domain.isin(["restaurant", "hotel", "food"])) & (df.language == "en")].copy()
rhf_en = rhf_en.reset_index(drop=True)
print("Combined rows:", rhf_en.shape)

# Dynamically pull real sentiment values instead of hardcoding — catches conflict
sentiments = sorted(rhf_en["sentiment"].dropna().unique())
print("Sentiments found:", sentiments)

rhf_categories = sorted(rhf_en["category"].dropna().unique())
rhf_label2id = {l: i for i, l in enumerate(
    ["O"] + [f"{p}-{c}-{s}" for c in rhf_categories for s in sentiments for p in ["B", "I"]]
)}
print("Restaurant+Hotel+Food labels:", len(rhf_label2id))

rhf_tagged = build_tagged_df(rhf_en, task="tasd")
rhf_tagged = rhf_tagged.reset_index(drop=True)

# Spot-check before committing GPU hours
non_null = rhf_en[rhf_en.aspect.notna() & (rhf_en.aspect.astype(str).str.strip() != "NULL")]
sample_check = rhf_tagged.loc[non_null.index[:50]]
unmatched = sum(1 for t in sample_check["tags"] if all(x == "O" for x in t))
print(f"Unmatched in 50-row sample: {unmatched} (investigate if > 5 before training)")

print(rhf_tagged["split"].value_counts())

Combined rows: (8410, 8)
Sentiments found: ['conflict', 'negative', 'neutral', 'positive']
Restaurant+Hotel+Food labels: 409
Unmatched in 50-row sample: 0 (investigate if > 5 before training)
split
train    4695
test     2516
dev      1199
Name: count, dtype: int64


In [12]:
# ============================================================
# Stage 2 (continue): Train XLM-R on Restaurant+Hotel+Food
# ============================================================
rhf_tok_fn = make_tokenize_fn(rhf_label2id)

train_ds_rhf = Dataset.from_pandas(
    rhf_tagged[rhf_tagged.split == "train"][["tokens", "tags"]]
).map(rhf_tok_fn, batched=True)

val_ds_rhf = Dataset.from_pandas(
    rhf_tagged[rhf_tagged.split == "dev"][["tokens", "tags"]]
).map(rhf_tok_fn, batched=True)

print("Train size:", len(train_ds_rhf), "| Val size:", len(val_ds_rhf))

collator = DataCollatorForTokenClassification(tokenizer)
model = AutoModelForTokenClassification.from_pretrained("xlm-roberta-base", num_labels=len(rhf_label2id))

args = TrainingArguments(
    output_dir="/content/xlmr_rhf",
    eval_strategy="epoch",
    save_strategy="no",              # standing rule — avoid disk/quota errors
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    weight_decay=0.01,
    logging_steps=50,
    fp16=True,
)

trainer_rhf = Trainer(
    model=model, args=args,
    train_dataset=train_ds_rhf, eval_dataset=val_ds_rhf,
    processing_class=tokenizer, data_collator=collator,
    compute_metrics=make_compute_metrics(list(rhf_label2id.keys())),
)
trainer_rhf.train()

# --- Save + verify ---
out_dir = "/content/drive/MyDrive/MABSA_Shared_Project/models/xlmr_restaurant_hotel_food_final"
os.makedirs(out_dir, exist_ok=True)
trainer_rhf.save_model(out_dir)
tokenizer.save_pretrained(out_dir)
with open(os.path.join(out_dir, "label_map.json"), "w") as f:
    json.dump(rhf_label2id, f)

check = AutoModelForTokenClassification.from_pretrained(out_dir)
print("Saved + verified:", check.config.num_labels == len(rhf_label2id))

Map:   0%|          | 0/4695 [00:00<?, ? examples/s]

Map:   0%|          | 0/1199 [00:00<?, ? examples/s]

Train size: 4695 | Val size: 1199


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.297358,0.312272,0.090713,0.355932,0.051980
2,0.247055,0.278022,0.180119,0.287263,0.131188
3,0.215892,0.258894,0.286718,0.331707,0.252475
4,0.180649,0.257456,0.296348,0.342532,0.261139
5,0.156672,0.249925,0.335826,0.372549,0.305693
6,0.147570,0.235833,0.317784,0.386525,0.269802
7,0.139693,0.242485,0.327354,0.413208,0.271040
8,0.115877,0.248931,0.343575,0.394231,0.304455
9,0.120891,0.243796,0.336047,0.418048,0.280941
10,0.112964,0.250679,0.371635,0.395804,0.350248


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Saved + verified: True


In [23]:
import os, json
from transformers import AutoModelForTokenClassification, AutoTokenizer, Trainer, DataCollatorForTokenClassification
import numpy as np
import evaluate

seqeval = evaluate.load("seqeval")

def make_tokenize_fn_safe(label2id_local, tok):
    def tokenize_and_align_labels(examples):
        tokenized_inputs = tok(examples["tokens"], truncation=True, is_split_into_words=True)
        all_labels = []
        for i, tags in enumerate(examples["tags"]):
            word_ids = tokenized_inputs.word_ids(batch_index=i)
            label_ids, prev = [], None
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != prev:
                    label_ids.append(label2id_local.get(tags[word_idx], -100))
                else:
                    label_ids.append(-100)
                prev = word_idx
            all_labels.append(label_ids)
        tokenized_inputs["labels"] = all_labels
        return tokenized_inputs
    return tokenize_and_align_labels

def make_compute_metrics(label_list_local):
    def compute_metrics(p):
        preds = np.argmax(p.predictions, axis=2)
        true_labels = [[label_list_local[l] for l in label if l != -100] for label in p.label_ids]
        true_preds = [[label_list_local[pp] for (pp, l) in zip(pred, label) if l != -100]
                      for pred, label in zip(preds, p.label_ids)]
        r = seqeval.compute(predictions=true_preds, references=true_labels)
        return {"f1": r["overall_f1"], "precision": r["overall_precision"], "recall": r["overall_recall"]}
    return compute_metrics

def eval_checkpoint_on_languages(ckpt_dir, languages, test_sets):
    model = AutoModelForTokenClassification.from_pretrained(ckpt_dir)
    tok = AutoTokenizer.from_pretrained(ckpt_dir)
    with open(os.path.join(ckpt_dir, "label_map.json")) as f:
        label2id = json.load(f)
    id2label = {v: k for k, v in label2id.items()}
    label_list = [id2label[i] for i in range(len(id2label))]

    tok_fn = make_tokenize_fn_safe(label2id, tok)
    collator = DataCollatorForTokenClassification(tok)
    trainer = Trainer(model=model, processing_class=tok, data_collator=collator,
                       compute_metrics=make_compute_metrics(label_list))

    per_lang_f1 = {}
    for lang in languages:
        tokenized_test = test_sets[lang].map(tok_fn, batched=True)
        preds = trainer.predict(tokenized_test)
        per_lang_f1[lang] = make_compute_metrics(label_list)(preds)["f1"]
        print(f"  {lang}: F1 = {per_lang_f1[lang]:.4f}")
    return per_lang_f1

In [24]:
print("=== Restaurant-only checkpoint ===")
restaurant_only_f1 = eval_checkpoint_on_languages(
    "/content/drive/MyDrive/MABSA_Shared_Project/models/xlmr_restaurant_only_final",
    languages, test_sets
)

print("\n=== Restaurant+Hotel+Food checkpoint ===")
rhf_f1 = eval_checkpoint_on_languages(
    "/content/drive/MyDrive/MABSA_Shared_Project/models/xlmr_restaurant_hotel_food_final",
    languages, test_sets
)

feature1_results = {
    lang: {"restaurant_only": restaurant_only_f1[lang], "restaurant_hotel_food": rhf_f1[lang]}
    for lang in languages
}
print("\n=== Feature 1 results ===")
print(feature1_results)

out_path = "/content/drive/MyDrive/MABSA_Shared_Project/results/rq3_feature1_domain_variety.json"
with open(out_path, "w") as f:
    json.dump(feature1_results, f, indent=2)
print(f"Saved to {out_path}")

=== Restaurant-only checkpoint ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Map:   0%|          | 0/788 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


  swahili: F1 = 0.2126


...

  spanish: F1 = 0.3699


Map:   0%|          | 0/789 [00:00<?, ? examples/s]

  german: F1 = 0.3687


Map:   0%|          | 0/793 [00:00<?, ? examples/s]

  thai: F1 = 0.1695

=== Feature 1 results ===
{'swahili': {'restaurant_only': np.float64(0.21264367816091953), 'restaurant_hotel_food': np.float64(0.16649949849548643)}, 'arabic': {'restaurant_only': np.float64(0.2765072765072765), 'restaurant_hotel_food': np.float64(0.270899470899471)}, 'hindi': {'restaurant_only': np.float64(0.30192307692307696), 'restaurant_hotel_food': np.float64(0.26556016597510373)}, 'vietnamese': {'restaurant_only': np.float64(0.15413891531874405), 'restaurant_hotel_food': np.float64(0.19627085377821396)}, 'spanish': {'restaurant_only': np.float64(0.36397748592870544), 'restaurant_hotel_food': np.float64(0.3698884758364312)}, 'german': {'restaurant_only': np.float64(0.3902877697841726), 'restaurant_hotel_food': np.float64(0.36867239732569246)}, 'thai': {'restaurant_only': np.float64(0.19714285714285712), 'restaurant_hotel_food': np.float64(0.1694915254237288)}}
Saved to /content/drive/MyDrive/MABSA_Shared_Project/results/rq3_feature1_domain_variety.json


In [25]:
# ============================================================
# FEATURE 2 — Step 1: Spot-check + normalize sentiment for
# Hotel, Laptop, Phone (English) before evaluating
# ============================================================

# Check sentiment values per domain
for dom in ["hotel", "laptop", "phone"]:
    vals = sorted(df[(df.domain == dom) & (df.language == "en")]["sentiment"].dropna().unique())
    print(f"{dom}: {vals}")

hotel: ['conflict', 'negative', 'neutral', 'positive']
laptop: ['negative', 'neutral', 'positive']
phone: ['negative', 'neutral', 'positive']


In [27]:
feature2_domains = ["hotel", "laptop", "phone"]
feature2_test_dfs = {}

for dom in feature2_domains:
    dom_df = df[(df.domain == dom) & (df.language == "en") & (df.split == "test")].copy()
    dom_df = dom_df.reset_index(drop=True)
    feature2_test_dfs[dom] = dom_df
    print(f"{dom}: {len(dom_df)} test rows")

hotel: 975 test rows
laptop: 757 test rows
phone: 1217 test rows


In [28]:
for dom in feature2_domains:
    dom_df = feature2_test_dfs[dom]
    dom_tagged = build_tagged_df(dom_df, task="tasd").reset_index(drop=True)

    non_null = dom_df[dom_df.aspect.notna() & (dom_df.aspect.astype(str).str.strip() != "NULL")]
    sample_idx = non_null.index[:50]
    sample_check = dom_tagged.loc[sample_idx]
    unmatched = sum(1 for t in sample_check["tags"] if all(x == "O" for x in t))
    print(f"{dom}: {unmatched}/{len(sample_idx)} unmatched")

hotel: 0/50 unmatched
laptop: 0/50 unmatched
phone: 2/50 unmatched


In [29]:
# ============================================================
# FEATURE 2 — Step 3: Evaluate Restaurant-only checkpoint
# on Hotel vs Laptop vs Phone (English)
# ============================================================
CKPT_RESTAURANT_ONLY = "/content/drive/MyDrive/MABSA_Shared_Project/models/xlmr_restaurant_only_final"

model_r = AutoModelForTokenClassification.from_pretrained(CKPT_RESTAURANT_ONLY)
tok_r = AutoTokenizer.from_pretrained(CKPT_RESTAURANT_ONLY)
with open(os.path.join(CKPT_RESTAURANT_ONLY, "label_map.json")) as f:
    label2id_r = json.load(f)
id2label_r = {v: k for k, v in label2id_r.items()}
label_list_r = [id2label_r[i] for i in range(len(id2label_r))]

tok_fn_r = make_tokenize_fn_safe(label2id_r, tok_r)
collator_r = DataCollatorForTokenClassification(tok_r)
trainer_r = Trainer(model=model_r, processing_class=tok_r, data_collator=collator_r,
                     compute_metrics=make_compute_metrics(label_list_r))

feature2_results = {}
for dom in feature2_domains:
    dom_df = feature2_test_dfs[dom]
    dom_tagged = build_tagged_df(dom_df, task="tasd").reset_index(drop=True)
    dom_ds = Dataset.from_pandas(dom_tagged[["tokens", "tags"]]).map(tok_fn_r, batched=True)
    preds = trainer_r.predict(dom_ds)
    feature2_results[dom] = make_compute_metrics(label_list_r)(preds)["f1"]
    print(f"{dom}: F1 = {feature2_results[dom]:.4f}")

print("\n=== Feature 2 results (Restaurant-only checkpoint) ===")
print(feature2_results)

out_path = "/content/drive/MyDrive/MABSA_Shared_Project/results/rq3_feature2_domain_similarity.json"
with open(out_path, "w") as f:
    json.dump(feature2_results, f, indent=2)
print(f"Saved to {out_path}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Map:   0%|          | 0/975 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


hotel: F1 = 0.2866


Map:   0%|          | 0/757 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


laptop: F1 = 0.0000


Map:   0%|          | 0/1217 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


phone: F1 = 0.0000

=== Feature 2 results (Restaurant-only checkpoint) ===
{'hotel': np.float64(0.28662420382165604), 'laptop': np.float64(0.0), 'phone': np.float64(0.0)}
Saved to /content/drive/MyDrive/MABSA_Shared_Project/results/rq3_feature2_domain_similarity.json


In [30]:
# Check: is Hotel's non-zero F1 driven by correctly predicting NULL/no-aspect tokens,
# or is the model actually getting some non-O predictions right?
hotel_tagged_check = build_tagged_df(feature2_test_dfs["hotel"], task="tasd").reset_index(drop=True)
non_o_tag_rows = sum(1 for tags in hotel_tagged_check["tags"] if any(t != "O" for t in tags))
print(f"Hotel test rows with at least one non-O tag: {non_o_tag_rows} / {len(hotel_tagged_check)}")

Hotel test rows with at least one non-O tag: 707 / 975


In [31]:
restaurant_cats = set(l.split("-")[1] for l in label2id_r if l != "O")

laptop_cats = set(df[(df.domain == "laptop") & (df.language == "en")]["category"].dropna().unique())
phone_cats = set(df[(df.domain == "phone") & (df.language == "en")]["category"].dropna().unique())

print(f"Restaurant categories in checkpoint: {len(restaurant_cats)}")
print(f"Laptop categories overlapping with restaurant: {len(restaurant_cats & laptop_cats)}")
print(f"Phone categories overlapping with restaurant: {len(restaurant_cats & phone_cats)}")

Restaurant categories in checkpoint: 13
Laptop categories overlapping with restaurant: 0
Phone categories overlapping with restaurant: 0
